# Tutorial: Harmonic Oscillator in ROS

```{admonition} Download
:class: tip
{download}`Download this notebook <harmonic_oscillator_in_ros.ipynb>`
```

## Overview
This is the introductory notebook for the dynamicalnodes pipeline. We will first model a toy dynamical system, the harmonic oscillator, using the DynamicalSystem class, and then implement the system as a lived ROS graph using the ROSNode class. This notebook will introduce a workflow whereby we first model our system as a discrete-time dynamical system.

# The Harmonic Oscillator

Recall that a simple harmonic oscillator has the following ODE:

$ \frac{d^2x}{dt^2} + \omega^2 x = \frac{F(t)}{m} $

where
- $ \frac{d^2x}{dt^2} $ is the acceleration term (second derivative of position  $x$  with respect to time  $t$ (often denoted $\ddot{x}$)
- $ \omega^2 x$ is the restoring force term, where $\omega = \sqrt{\frac{k}{m}}$ is the natural angular frequency
- $\frac{F(t)}{m}$ is the external forcing function $F(t)$ scaled by the mass  $m$  (often denoted as  just $f(t)$

To rewrite this in a continuous-time state-space form, we define a state vector $(x_1,x_2)$, where $x_1$ is position and $x_2 := \dot{x}_1$ is velocity.  We then have

$ \frac{d}{dt} \begin{bmatrix} x_1 \\ x_2\end{bmatrix} = \begin{bmatrix} x_2 \\ \frac{F(t)}{m} - \omega^2 x_1\end{bmatrix}$

Assuming an unforced osclilator (i.e. $F(t) = 0$), we have a linear system

$ \frac{d}{dt} \begin{bmatrix} x_1 \\ x_2\end{bmatrix} = \begin{bmatrix} 0 & 1 \\  - \omega^2 & 0 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2\end{bmatrix}$

Setting a time step $\Delta t$, we convert this to a discrete-time system via Euler integration:

$ \begin{bmatrix} x_1 \\ x_2\end{bmatrix}_{k+1} = \begin{bmatrix} 0 & \Delta t \\  - \omega^2 \Delta t & 0 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2\end{bmatrix}_k$

Set $\vec{x} = \begin{bmatrix} x_1 \\ x_2\end{bmatrix}$. Assume we have direct access to velocity measurements for our system, given by the measurement function $h(\vec{x}) = x_2$. We then have the following discrete-time state-space representation:

$\begin{align} \dot{\vec{x}} &= f(\vec{x}) = \begin{bmatrix} 0 & \Delta t \\  - \omega^2 \Delta t & 0 \end{bmatrix} \vec{x} \\ y &= h(\vec{x}) = \begin{bmatrix} 0 & 1 \end{bmatrix} \vec{x} \end{align}$

where $f$ is our dynamics function, $h$ is our measurement function, and $y$ is our measurement. The discrete-time formulation maps directly onto the inherently discrete software representation, which means we are now ready to model this systing using the `DynamicalSystem` class!

In [ ]:
import dynamicalnodes
import numpy as np
from dynamicalnodes import DynamicalSystem


# sho = simple harmonic oscillator
def f_sho(xk: dict, dt: float = 0.1, omega: float = 1) -> dict:
    x1, x2 = xk["x1"], xk["x2"]  # unpack dictionary
    x = np.array([x1, x2])  # state vector (2,1)
    Fk = np.array([[0, dt], [-(omega**2) * dt, 0]])  # dynamics matrix (2,2)
    x_next = Fk @ x  # update state
    return {"x1": x_next[0], "x2": x_next[1]}


def h_sho(xk: dict) -> float:
    x1, x2 = xk["x1"], xk["x2"]  # unpack dictionary
    x = np.array([x1, x2])  # state vector (2,1)
    Hk = np.array([0, 1])  # measurement matrix (1,2)
    yk = float(Hk @ x)  # compute measurement
    return yk


def h_sho_simple(xk: dict) -> float:
    return xk["x2"]  # just return x2 directly; no need for matrix mult


sho = DynamicalSystem(f=f_sho, h=h_sho)

print(sho)

Note that we took some liberties in implementing our dynamics function; we added `dt` and `omega` as optional parameters. This lets us change the behaviour of our system without having to redefine our functions. We can now simulate this  system in a for-loop.

In [ ]:

xk = {"x1": 1, "x2": 0}  # initial state dictionary

states = []
measurements = []
hz = 0.1
for tk in np.arange(0, 10, hz):
    xknext, yk = sho.eval(xk=xk, dt=0.1, omega=1)  # eval (f,h) on all parameters

    # save history
    states.append(xk)
    measurements.append(yk)

    xk = xknext  # update stat
    print(xk)

import matplotlib.pyplot

plot(states, measurements)

We can now model this in ROS. There is no input to this dynamical system, so in essence, this is just a signal generating ROS node. Since the output measurement, yk, is of Python type float, we will need to create a translation function to translate float to Float64, the corresponding ROS data type.

In [ ]:
from dynamicalnodes import ROSNode

ros_hz = 1 / hz
sho_node = ROSNode(dynamical_system=sho)

We can test our translation pipelines are working properly by simulating the system (this will be more important when there are multiple translation functions per node i.e. multiple subscriptions)

Next, we write this to a rclpy file, specifing a hz at which this node runs (corresponding to 1/hz in the for-loop above) as well as an initial state. This is runtime-specific metadata, which is why it is included in the final writing step.

In [ ]:
sho_node.write_to_rclpy(hz=ros_hz)

Finally, we run the node. Run the node using [blank]. You can see the node using ros node. You can see the topic yk using ros echo. To see a live plot, we recommend using plotjuggler (see installation isntructions).

We can specify dt and omega as dynamic parameters, and these can now be changed dynamically using ros set (see how it affects the plot).

We can even specify the state of the system as a dynamic parameter (after all, programmatically, xk is nothing more than another function parameter). To do so, xk is a dict, so (actually, can we do this?)

Tada! That wasn't so hard. Now let's do with a forcing input. We will need to come up with a dynamical system that generates a forcing term, and rewrite our first system to accept a force input. Lets consider a sinusoidal signal generator. (do math derivation here, and then follow patter above)

Lets get more comples: let's model a coupled system. We update our current dynamical system to take an input u in Newtons, and then create a copy of that. We now have two dynamical systems influencing eachother. We follow the same precedure as above: starting with the discrete-time formulation, etc...

Proceed to the car_cruise_control tutorial to do an entire control system using proper cnotrollers and estimators in ROS.